# PyTorch 神经网络基础

## 模型构造

层和块

In [23]:
import torch
from torch import nn
from torch.nn import functional as F
# nn.Sequential 是一个Module，会按顺序执行层
net = nn.Sequential(
    nn.Linear(20,256),  #全连接层  输入20，输出256
    nn.ReLU(),          #激活函数
    nn.Linear(256,10))  #全连接层  输入256，输出10

x=torch.randn(2,20)  # 2个样本，一个样本20个特征

net(x)

tensor([[-0.9180, -0.4674, -0.1559, -0.0010, -0.6322,  0.1741,  0.3200, -0.0399,
         -0.0433,  0.0402],
        [-0.1292, -0.1501, -0.1973,  0.1210, -0.1590,  0.1280, -0.0397,  0.0311,
          0.3618,  0.3278]], grad_fn=<AddmmBackward0>)

In [24]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()     #父类初始化
        self.hidden = nn.Linear(20,256)
        self.out = nn.Linear(256,10)
    def forward(self,x):
        return self.out(F.relu(self.hidden(x)))

In [25]:
net = MLP()
net(x)

tensor([[-0.2072, -0.0006, -0.0951, -0.4196,  0.2387,  0.1898, -0.0184, -0.2112,
         -0.3065,  0.3390],
        [ 0.0181,  0.1753, -0.1926, -0.3016, -0.0761, -0.0350, -0.2346, -0.2189,
          0.0770,  0.0090]], grad_fn=<AddmmBackward0>)

顺序块

In [26]:
class MySequential(nn.Module):
    def __init__(self,*args): 
    # *是打包符，将传入多个参数打包成一个元组
        super().__init__()
        for block in args:
            self._modules[block]=block
            # _modules[]是一个有序词典，确保顺序
    def forward(self,x):
        for block in self._modules.values():
            x=block(x)
        return x

net = MySequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
net(x)

tensor([[-0.4907, -0.4219, -0.1239, -0.4984, -0.5104, -0.4187,  0.1662,  0.5343,
          0.2785,  0.2632],
        [ 0.1225, -0.2238, -0.2632, -0.2499, -0.7561, -0.0885,  0.2143,  0.0707,
          0.1825,  0.3487]], grad_fn=<AddmmBackward0>)

在正向传播函数中执行代码

In [27]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 放入随机权重
        self.rand_weight = torch.rand((20,20),requires_grad=False)
        self.linear = nn.Linear(20,20)
    #比较灵活可以自己随便写
    def forward(self,x):
        x=self.linear(x)
        x=F.relu(torch.mm(x,self.rand_weight)+1)  # 偏置为1
        x=self.linear(x)
        while x.abs().sum()>1:
            x/=2
        return x.sum()

net = FixedHiddenMLP()
net(x)

tensor(0.2119, grad_fn=<SumBackward0>)

混合搭配各种组合快

In [28]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU())
        self.linear = nn.Linear(32,16)
    def forward(self,x):
        return self.linear(self.net(x))

chimera = nn.Sequential(NestMLP(),nn.Linear(16,20),FixedHiddenMLP())
chimera(x)

tensor(0.2441, grad_fn=<SumBackward0>)

## 参数管理

单隐藏层的多层感知机

In [29]:
import torch
from torch import nn
# Sequential 相当于一个list
net = nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,1))
X=torch.rand(size=(2,4))
net(X)

tensor([[-0.2179],
        [-0.1261]], grad_fn=<AddmmBackward0>)

In [30]:
print(net[2].state_dict())
# 权重和偏移

OrderedDict([('weight', tensor([[ 0.2209,  0.1629,  0.1846,  0.0164, -0.0219,  0.2714,  0.3188,  0.1674]])), ('bias', tensor([-0.3456]))])


In [31]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)  # .data 访问数据本身

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([-0.3456], requires_grad=True)
tensor([-0.3456])


In [32]:
net[2].weight.grad == None

True

一次访问所有参数

In [33]:
print(*[(name,param.shape) for name,param in net[0].named_parameters()])
# 拿出所有参数，1是RELU不显示
print(*[(name,param.shape) for name,param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


从嵌套块收集参数

In [34]:
def block1():
    return nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,4))
# block2嵌套4个block1
def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block{i}',block1())
    return net

rgnet = nn.Sequential(block2(),nn.Linear(4,1))
rgnet(X)

tensor([[-0.0313],
        [-0.0324]], grad_fn=<AddmmBackward0>)

In [35]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


内置初始化

In [36]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight,mean=0,std=0.01)  #normal正态分布
        nn.init.zeros_(m.bias)

net.apply(init_normal)   #递归将函数应用于当前模块（及其子模块）
net[0].weight.data[0],net[0].bias.data[0]

(tensor([ 0.0042,  0.0094, -0.0048,  0.0073]), tensor(0.))

In [37]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight,1)  #常为1
        nn.init.zeros_(m.bias)

net.apply(init_constant)  
net[0].weight.data[0],net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

对某些块应用不同的初始化方法

In [38]:
def xavier(m):  #让每一层输出的方差尽量等于输入的方差
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)  #均匀分布中采样
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight,42)

net[0].apply(xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([ 0.3349, -0.7062, -0.6950, -0.6832])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


自定义初始化

In [39]:
def my_init(m):  
    if type(m) == nn.Linear:
        print(
            "Init",
            *[(name,param.shape) for name,param in m.named_parameters()][0])
        nn.init.uniform_(m.weight,-10,10)
        m.weight.data*=  m.weight.data.abs()>=5  #小于5就置为0

net[0].apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])


tensor([[-0.0000, -0.0000,  9.3932,  6.3220],
        [ 5.7695,  6.6501, -5.4857,  7.7538]], grad_fn=<SliceBackward0>)

In [40]:
net[0].weight.data[:]+=1
net[0].weight.data[0,0] = 42
net[0].weight.data[0]

tensor([42.0000,  1.0000, 10.3932,  7.3220])

参数绑定

In [41]:
shared = nn.Linear(8,8)
net = nn.Sequential(nn.Linear(4,8),nn.ReLU(),shared,nn.ReLU(),shared,nn.ReLU(),nn.Linear(8,1))
net(X)
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0,0] = 100
print(net[2].weight.data[0]==net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


## 自定义层

构造一个没有任何参数的自定义层

In [42]:
import torch
import torch.nn.functional as F
from torch import nn

class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,X):
        return X-X.mean()
    
layer =CenteredLayer()
layer(torch.FloatTensor([1,2,3,4,5]))

tensor([-2., -1.,  0.,  1.,  2.])

将层作为组件合并到更为复杂的模型中

In [43]:
net =nn.Sequential(nn.Linear(8,128),CenteredLayer())

Y=net(torch.rand(4,8))
Y.mean()

tensor(-6.5193e-09, grad_fn=<MeanBackward0>)

带参数的图层

In [49]:
X,X.shape

(tensor([[0.7672, 0.0928, 0.8925, 0.5047],
         [0.6241, 0.4569, 0.7396, 0.7436]]),
 torch.Size([2, 4]))

In [51]:
class MyLinear(nn.Module):
    def __init__(self, in_units,units):
    # in_units,units是输入维度和输出维度
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units,units))
        self.bias = nn.Parameter(torch.randn(units,))

    def forward(self,X):
        linear = torch.matmul(X,self.weight)+self.bias
        return F.relu(linear)
    
dense = MyLinear(4,3)
dense.weight

Parameter containing:
tensor([[-0.0147,  1.6048, -0.0345],
        [-1.2075,  0.9165,  1.9228],
        [ 0.4378,  2.7224,  0.9201],
        [-1.2223, -0.7820,  0.2208]], requires_grad=True)

In [45]:
dense(torch.rand(2,5))

tensor([[0.6697, 1.0182, 0.7109],
        [0.8339, 0.3899, 0.0000]])

In [46]:
net = nn.Sequential(MyLinear(64,8),MyLinear(8,1))
net(torch.rand(2,64))

tensor([[ 6.1713],
        [18.1561]])

## 读写文件

加载和保存张量

In [53]:
import torch
from torch import nn
from torch.nn import functional as Flatten

x = torch.arange(4)
torch.save(x,'x-file')

x2=torch.load("x-file")
x2

tensor([0, 1, 2, 3])

In [54]:
y = torch.zeros(4)
torch.save([x,y],'x-file')
x2,y2 = torch.load('x-file')
(x2,y2)

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

In [55]:
mydict = {'x':x,'y':y}
torch.save(mydict,'mydict')
mydict2 = torch.load('mydict')
mydict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

加载和保存模型参数

In [56]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20,256)
        self.output = nn.Linear(256,10)
    def forward(self,x):
        return self.output(F.relu(self.hidden(x)))
    
net = MLP()
X = torch.randn(size=(2,20))
Y = net(X)

In [57]:
torch.save(net.state_dict(),'mlp.params')

In [59]:
clone = MLP()
clone.load_state_dict(torch.load('mlp.params'))
clone.eval()

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

In [60]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])